#Ingest Races File

1. Read the file using spark dataframe reader API
2. Add Metadata Column
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table



In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-Common/01.environment-config

In [0]:
%run ../00-Common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"


### Step 1 - Read the CSV file using the dataframe reader API


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
races_schema = StructType([
    StructField('season', IntegerType()),
    StructField('round', IntegerType()),
    StructField('url', StringType()),
    StructField('raceName', StringType()),
    StructField('date', DateType()),
    StructField('circuitId', StringType())
])

In [0]:
races_df = (
        spark.read.format('csv')
        .option('header', 'true')
        # .option('inferSchema','true')
        .option('mode','FAILFAST')
        .schema(races_schema)
        .load(source_file)
    )


### Step 2 - Add Metadata Columns

1. Source File
2. Ingestion Timestamp

In [0]:
races_final_df = add_ingestion_metadata(races_df)


### Step 3 - Write to bronze delta table

In [0]:
write_to_bronze(input_df = races_final_df, table_name = table_name, batch_id = v_batch_id)

In [0]:
%sql
SELECT * FROM formula1.bronze.races;